# Generator — бэйзлайн (дефолтные параметры)

Алгоритмы запускаются с параметрами по умолчанию на всех N_REPS репликациях каждого комбо.

In [1]:
import sys, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    adjusted_rand_score, adjusted_mutual_info_score,
    normalized_mutual_info_score, fowlkes_mallows_score,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
)

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')


In [2]:
from algorithms import (
    DBSCANWrapper, HDBSCANWrapper,
    DPCWrapper, RDDACWrapper, CKDPCWrapper,
)

ALG_NAMES = ['DBSCAN', 'HDBSCAN', 'DPC', 'RD-DAC', 'CKDPC']
ALG_CLASSES = {
    'DBSCAN':  DBSCANWrapper,
    'HDBSCAN': HDBSCANWrapper,
    'DPC':     DPCWrapper,
    'RD-DAC':  RDDACWrapper,
    'CKDPC':   CKDPCWrapper,
}


In [3]:
from pathlib import Path

EXPERIMENT_DIR = Path('../synthetic_datasets/experiment')
N_REPS = 10

def parse_combo(name):
    parts = {}
    for p in name.split('_'):
        if p.startswith('V='): parts['V'] = int(p[2:])
        if p.startswith('K='): parts['K'] = int(p[2:])
        if p.startswith('alpha='): parts['alpha'] = float(p[6:])
    return parts

combos = sorted([d for d in EXPERIMENT_DIR.iterdir() if d.is_dir()], key=lambda p: p.name)
print(f"Found {len(combos)} combos")


Found 18 combos


In [4]:
def compute_all_metrics(X, labels, y_true):
    labels  = np.asarray(labels, dtype=int)
    y_true  = np.asarray(y_true, dtype=int)
    mask_nn = labels != -1
    k_found   = len(set(labels[mask_nn].tolist())) if mask_nn.any() else 0
    noise_pct = float((~mask_nn).sum()) / len(labels) * 100
    if k_found >= 2:
        ari = float(adjusted_rand_score(y_true, labels))
        ami = float(adjusted_mutual_info_score(y_true, labels))
        nmi = float(normalized_mutual_info_score(y_true, labels))
        fmi = float(fowlkes_mallows_score(y_true, labels))
    elif k_found == 1:
        ari = ami = nmi = fmi = 0.0
    else:
        ari = ami = nmi = fmi = float('nan')
    X_sub, l_sub = X[mask_nn], labels[mask_nn]
    if mask_nn.sum() >= 2 and len(np.unique(l_sub)) >= 2:
        try:    sc  = float(silhouette_score(X_sub, l_sub))
        except: sc  = float('nan')
        try:    chi = float(calinski_harabasz_score(X_sub, l_sub))
        except: chi = float('nan')
        try:    dbi = float(davies_bouldin_score(X_sub, l_sub))
        except: dbi = float('nan')
    else:
        sc = chi = dbi = float('nan')
    return dict(k_found=k_found, noise_pct=noise_pct,
                ARI=ari, AMI=ami, NMI=nmi, FMI=fmi, SC=sc, CHI=chi, DBI=dbi)


def run_default(alg_class, X):
    try:
        return np.asarray(alg_class().fit_predict(X), dtype=int)
    except Exception as e:
        print(f'  ERROR: {e}')
        return np.zeros(len(X), dtype=int)


In [6]:
ALL_RESULTS = {}

for combo_dir in combos:
    cname = combo_dir.name
    p = parse_combo(cname)
    V, K, alpha = p['V'], p['K'], p['alpha']
    rep_files = sorted(combo_dir.glob('rep*_X.npy'))[:N_REPS]
    per_rep = {a: [] for a in ALG_NAMES}

    for xf in rep_files:
        rep_id = xf.stem.replace('_X', '')
        X  = np.load(xf)
        rf = np.load(combo_dir / f'{rep_id}_rf.npy')
        y  = rf.astype(int) - 1
        for alg_name in ALG_NAMES:
            try:
                lbl  = np.asarray(ALG_CLASSES[alg_name]().fit_predict(X), dtype=int)
                mets = compute_all_metrics(X, lbl, y)
            except Exception:
                mets = dict(k_found=0, noise_pct=100.0, ARI=float('nan'),
                            AMI=float('nan'), NMI=float('nan'), FMI=float('nan'),
                            SC=float('nan'), CHI=float('nan'), DBI=float('nan'))
            per_rep[alg_name].append(mets)

    ALL_RESULTS[cname] = {'V': V, 'K': K, 'alpha': alpha, 'per_rep': per_rep}


In [7]:
def _safe_mean(vals, key):
    v = [x[key] for x in vals if isinstance(x[key], (int, float)) and x[key]==x[key]]
    return round(float(np.mean(v)), 4) if v else float('nan')

METS = ['ARI', 'AMI', 'NMI', 'FMI', 'SC', 'DBI', 'k_found', 'noise_pct']
rows_avg = []
for cname, res in ALL_RESULTS.items():
    for alg in ALG_NAMES:
        vals = res['per_rep'][alg]
        row = {'combo': cname, 'V': res['V'], 'K': res['K'], 'alpha': res['alpha'], 'Algorithm': alg}
        for m in METS:
            row[m] = _safe_mean(vals, m)
        rows_avg.append(row)
df_avg = pd.DataFrame(rows_avg)

pivot = df_avg.pivot_table(index=['V','K','alpha'], columns='Algorithm', values='ARI', aggfunc='mean')
pivot = pivot[ALG_NAMES]
display(pivot.style
        .format('{:.4f}', na_rep='-')
        .background_gradient(cmap='RdYlGn', axis=None, vmin=0, vmax=1)
        .set_caption('Средний ARI (дефолтные параметры)'))
